# XAUUSDzero D1 breakout research

Separate EMA and confirmed-swing event cohorts. This notebook uses the eight registered signal-time predictors and direction only. OHLC/raw prices, timestamps, entry/exit fields, MAE/MFE, returns and outcomes remain audit data, never predictors. Fixed 2R labels use a 30-bar horizon, ATR 20, buffer 0.05, zero slippage and zero commission. Net R therefore omits trading costs. MAE/MFE are descriptive R-unit excursions over the full available observation window, including after an early exit. Outcomes are observational backtest labels from one market and do not establish live profitability.

In [ ]:
import hashlib
import importlib
import json
import math
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.tree import export_text

repo = Path.cwd() if (Path.cwd() / "app").is_dir() else Path.cwd().parent
sys.path.insert(0, str(repo))
research = importlib.import_module("ML.research")
STRATEGIES = research.STRATEGIES
PREDICTORS = research.PREDICTORS
export_events = research.export_events
eligible_events = research.eligible_events
chronological_split = research.chronological_split
evaluate_models = research.evaluate_models
predictors = research.predictors

source = Path(
    os.environ.get(
        "BREAKOUT_SOURCE", repo.parent / "data analysis" / "mt5_data" / "XAUUSDzero_D1_max_bars.csv"
    )
)
output = repo / "exports/ML"
paths = {name: output / f"{name}.parquet" for name in STRATEGIES}
if not all(path.exists() for path in paths.values()):
    if any(path.exists() or path.with_suffix(".parquet.json").exists() for path in paths.values()):
        raise FileExistsError(
            "Incomplete research export: move the existing files before regenerating"
        )
    paths = export_events(source, output)
fingerprint = hashlib.sha256(source.read_bytes()).hexdigest()
frames = {}
for name, path in paths.items():
    metadata = json.loads(Path(str(path) + ".json").read_text(encoding="utf-8"))
    if metadata["source_sha256"] != fingerprint:
        raise ValueError(
            f"{name}: source fingerprint differs; move old export files before regenerating"
        )
    if metadata["model_predictors"] != list(PREDICTORS):
        raise ValueError(f"{name}: export predictor contract differs")
    frames[name] = pd.read_parquet(path)
print("Source:", source, "SHA-256:", fingerprint)
display(
    pd.DataFrame(
        {
            name: {"events": len(frame), "unique_ids": frame.id.nunique()}
            for name, frame in frames.items()
        }
    ).T
)

## Event accounting and outcome uncertainty

Eligibility and exclusions are counted before model selection. Later Wilson 95% intervals describe event-level target-hit proportions; overlapping event windows reduce independence, so treat intervals as descriptive. Ambiguous, censored, unfilled, invalid-entry, incomplete-horizon and missing-predictor rows remain in the audit exports but leave modeling.

In [ ]:
def wilson(successes: int, count: int) -> tuple[float, float]:
    if count == 0:
        return (float("nan"), float("nan"))
    z = 1.96
    center = (successes / count + z * z / (2 * count)) / (1 + z * z / count)
    half = (
        z
        * math.sqrt(
            (successes / count) * (1 - successes / count) / count + z * z / (4 * count * count)
        )
        / (1 + z * z / count)
    )
    return center - half, center + half


eligible = {}
for name, frame in frames.items():
    print("\n", name)
    display(
        frame.loc[~frame.eligible]
        .groupby(["label_status", "horizon_complete"], dropna=False)
        .size()
        .rename("excluded_events")
        .to_frame()
    )
    selected = eligible_events(frame)
    eligible[name] = selected
    print(
        "Model eligible:",
        len(selected),
        "of",
        len(frame),
        "; excluded:",
        len(frame) - len(selected),
    )
    print("Missing predictors among otherwise eligible:", int(frame.eligible.sum()) - len(selected))

## Chronological modeling

For each strategy, the final 20% of eligible events is untouched during selection. Development is purged at the test boundary. Three expanding validation folds purge training events whose planned 30-bar outcome end reaches validation start. No fitting, binning or threshold selection uses test outcomes. Small samples or single-class folds remain descriptive only.

In [ ]:
results = {}
for name, cohort in eligible.items():
    result = evaluate_models(cohort)
    results[name] = result
    print(
        "\n",
        name,
        "development:",
        result["development_count"],
        "test:",
        result["test_count"],
        "test-boundary purge:",
        result["test_boundary_purged"],
        "fold purges:",
        result["fold_purged"],
    )
    if "unavailable" in result:
        print("Model unavailable:", result["unavailable"])
        continue
    comparison = pd.DataFrame(
        {"CV Brier": result["mean_cv_brier"], "untouched test Brier": result["all_test_brier"]}
    )
    display(comparison.sort_values("CV Brier"))
    print(
        "CV selected:",
        result["selected"],
        "; XGBoost device:",
        result["device"],
        ";",
        result["gpu_note"],
    )
    print("Shallow tree rules (fit on purged development only):")
    print(export_text(result["tree_model"], feature_names=list(predictors(cohort).columns)))
    development, _, _ = chronological_split(cohort)
    leaves = result["tree_model"].apply(predictors(development))
    display(
        pd.DataFrame({"leaf": leaves, "hit": development.target_first.astype(int)})
        .groupby("leaf")
        .hit.agg(["size", "mean"])
        .rename(columns={"size": "development_support", "mean": "target_hit_rate"})
    )

## Validation-selected favorable conditions and untouched-test assessment

The favorable condition is a probability cutoff at the top 20%, 40%, or 60% of out-of-fold validation scores, chosen by validation lift, with at least 10 validation events. It is assessed once on test. A missing qualifying cutoff is reported rather than forced.

In [ ]:
for name, result in results.items():
    if "unavailable" in result:
        continue
    cohort = eligible[name]
    development, _, _ = chronological_split(cohort)
    oof = result["oof"]
    truth = development.target_first.iloc[oof.development_index].astype(int).to_numpy()
    probabilities = oof.probability.to_numpy()
    prevalence = truth.mean()
    choices = []
    for top_fraction in (0.2, 0.4, 0.6):
        cutoff = float(np.quantile(probabilities, 1 - top_fraction))
        selected = probabilities >= cutoff
        if selected.sum() >= 10 and prevalence > 0:
            choices.append((truth[selected].mean() / prevalence, cutoff, int(selected.sum())))
    test = result["test_events"]
    test_truth = test.target_first.astype(int).to_numpy()
    test_prob = result["test_probability"]
    print("\n", name, "validation prevalence:", prevalence, "test prevalence:", test_truth.mean())
    if choices:
        lift, cutoff, support = max(choices)
        selected = test_prob >= cutoff
        print("Validation selected cutoff:", cutoff, "support:", support, "lift:", lift)
        print(
            "Test condition support:",
            int(selected.sum()),
            "hit rate:",
            float(test_truth[selected].mean()) if selected.any() else float("nan"),
            "test lift:",
            float(test_truth[selected].mean() / test_truth.mean())
            if selected.any() and test_truth.mean()
            else float("nan"),
        )
    else:
        print("No cutoff met validation support")
    val_precision, val_recall, _ = precision_recall_curve(truth, probabilities)
    test_precision, test_recall, _ = precision_recall_curve(test_truth, test_prob)
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    for label, observed, forecast in (
        ("CV", truth, probabilities),
        ("test", test_truth, test_prob),
    ):
        true_bin, predicted_bin = calibration_curve(
            observed, forecast, n_bins=5, strategy="quantile"
        )
        axes[0].plot(predicted_bin, true_bin, marker="o", label=label)
    axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[0].set(title=f"{name}: calibration", xlabel="Mean forecast", ylabel="Observed hit rate")
    axes[0].legend()
    axes[1].plot(
        val_recall,
        val_precision,
        label=f"CV AP={average_precision_score(truth, probabilities):.2f}",
    )
    axes[1].plot(
        test_recall,
        test_precision,
        label=f"Test AP={average_precision_score(test_truth, test_prob):.2f}",
    )
    axes[1].set(title="Precision–recall", xlabel="Recall", ylabel="Precision")
    axes[1].legend()
    for label, observed, forecast in (
        ("CV", truth, probabilities),
        ("test", test_truth, test_prob),
    ):
        order = np.argsort(-forecast)
        ordered = observed[order]
        count = np.arange(1, len(ordered) + 1)
        axes[2].plot(count / len(count), np.cumsum(ordered) / count / observed.mean(), label=label)
    axes[2].axhline(1, linestyle="--", color="gray")
    axes[2].set(title="Cumulative lift", xlabel="Top fraction by probability", ylabel="Lift")
    axes[2].legend()
    fig.tight_layout()
    plt.show()

## Descriptive outcomes

Feature-bin hit rates use purged development only; all-event distributions are displayed after model selection and test assessment. Net R uses the labeler's zero-cost defaults. MAE/MFE cover the full observation horizon, including after early exits, so they describe path exposure and are not realized trade outcomes or predictors.

In [ ]:
for name, frame in frames.items():
    print("\n", name, "all event statuses and R distributions")
    display(
        frame.groupby(["label_status", "horizon_complete"], dropna=False)
        .size()
        .rename("events")
        .to_frame()
    )
    selected = eligible[name]
    hits = int(selected.target_first.sum())
    print("Target first:", hits, "/", len(selected), "; Wilson 95%:", wilson(hits, len(selected)))
    display(frame[["net_r", "mae_r", "mfe_r"]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T)
    cohort = eligible[name].copy()
    if cohort.empty:
        continue
    cohort["year"] = pd.to_datetime(cohort.signal_time, unit="s", utc=True).dt.year
    for grouping in ("year", "direction"):
        grouped = cohort.groupby(grouping, observed=True).agg(
            events=("id", "size"),
            hits=("target_first", "sum"),
            mean_net_r=("net_r", "mean"),
            median_mae_r=("mae_r", "median"),
            median_mfe_r=("mfe_r", "median"),
        )
        grouped["hit_rate"] = grouped.hits / grouped.events
        display(grouped)
    development_bins, _, _ = chronological_split(cohort)
    for feature in PREDICTORS[:-1]:
        bins = pd.qcut(development_bins[feature].astype(float), q=4, duplicates="drop")
        grouped = development_bins.groupby(bins, observed=True).target_first.agg(["size", "mean"])
        print(feature)
        display(grouped.rename(columns={"size": "events", "mean": "target_hit_rate"}))
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.3))
    for axis, column in zip(axes, ("net_r", "mae_r", "mfe_r")):
        axis.hist(frame[column].dropna().astype(float), bins=30)
        axis.set(title=f"{name}: {column}", xlabel="R", ylabel="Events")
    fig.tight_layout()
    plt.show()

## Interpretation

These are event-level associations in one historical XAUUSDzero D1 source. Overlapping events, detector choices, fixed barriers, intrabar ambiguity, data quality, execution costs, spread and regime change limit causal and forward claims. The untouched test is a single historical period. Investigate stability and realistic costs before any trading use.